# Inductance Matrix of Coupled Coils (Magnetostatic)

`gsim.palace.MagnetostaticSim` drives a unit current around each conductor loop in turn, and Palace returns the
**inductance matrix** between the loops from the stored magnetic energy. This notebook extracts it for two
single-turn coils of the IHP SG13G2 PDK (`inductor2`):

1. a **coupled pair**, one coil on TopMetal2 over a copy on TopMetal1, which shows the workflow;
2. a **side-by-side pair** on TopMetal2, whose mutual inductance is checked against an analytic reference.

**Requirements:**
- `ihp-gdsfactory==2.0.1`
- A Palace backend: a [GDSFactory+](https://gdsfactory.com) account for cloud simulation (`sim.run()`), or a local
  `palace` executable (`sim.run_local()`).

## Model and material overrides

The IHP material values are unchanged; the model makes these choices:

- **Conductors are perfect conductors.** Palace's magnetostatic solver carries current only on perfect-conductor
  surfaces, so the metal conductivities are not used. The result is the **external inductance**, the high-frequency
  limit: the current flows on the metal surfaces, with no internal inductance and no resistance.
- **No magnetic materials.** The IHP material cards carry no permeability, so μr = 1 everywhere.
- **No substrate.** `set_stack()` leaves it out by default. Doped silicon is not magnetic, and a static solve has no
  eddy currents, so it would only add mesh.
- **No guard ring, shield or fill.** A closed metal loop that carries no source screens the field.
  `inductor2` draws none; its marker layers (`INDdrawing`, the nofill layers) are not meshed.

The cell's `c.info["inductance"]` (33.3 pH) is a value passed through to the schematic, not a model of the
geometry, so it is not a reference here.

In [ ]:
import json
from pathlib import Path

import gdsfactory as gf
import numpy as np
from ihp import PDK, cells

from gsim.palace import MagnetostaticSim

PDK.activate()


@gf.cell
def inductor2_pair(pitch: float = 39.35, stacked: bool = False) -> gf.Component:
    """Two single-turn inductor2 coils, ports A_P1/A_P2 and B_P1/B_P2.

    Coil A is on TopMetal2. With stacked=True, coil B is a TopMetal1 copy rotated
    180 degrees about the octagon centre, coaxial with A; otherwise it is a
    TopMetal2 copy `pitch` um to the right.
    """
    c = gf.Component()
    a = c << cells.inductor2()
    if stacked:
        b = c << cells.inductor2(
            layer_metal_2="TopMetal1drawing", layer_metal_2_pin="TopMetal1pin"
        )
        b.rotate(180, center=(0, 44.675))
    else:
        b = c << cells.inductor2()
        b.movex(pitch)
    c.add_ports(a.ports, prefix="A_")
    c.add_ports(b.ports, prefix="B_")
    return c

## Coupled coils

Coil B sits 5.3 µm below coil A and is rotated by 180°, so its leads run clear of A's. Each coil gets one current
source between its two leads: a unit current enters at `P1`, runs round the coil and leaves at `P2`. gsim closes the
loop with a sheet across the 2.1 µm gap between the lead tips.

In [ ]:
coupled = inductor2_pair(stacked=True)
coupled.plot()

In [ ]:
sim = MagnetostaticSim()
sim.set_output_dir("./palace-sim-magnetostatic")
sim.set_geometry(coupled)
sim.set_stack()
# Keep the outer wall far from the coils: it biases the inductance.
sim.set_airbox(margin_x=100, margin_y=100, z_above=150, z_below=150)
sim.add_current_source("A", plus_port="A_P1", minus_port="A_P2")
sim.add_current_source("B", plus_port="B_P1", minus_port="B_P2")

for source in sim.resolve_current_sources():
    print(source.name, source.layer, source.center, source.direction)
print(sim.validate_config())

The mesh is graded from 1 µm at the metal (half the 2 µm trace width) to 30 µm in the air. `fmax` has no effect on a
static solve.

In [ ]:
sim.mesh(refined_mesh_size=1.0, max_mesh_size=30.0)

## Run and read the matrix

`check_cache=True` reuses a finished cloud job with the same inputs. `load_inductance` labels the matrix with the
source names, and reads Palace's log to refuse a solve that did not converge, since such a solve still writes a
matrix that can look plausible. `problems()` lists what is physically inconsistent (reciprocity, energy, |k| ≤ 1); it
is empty here.

In [ ]:
results = sim.run(check_cache=True)
inductance = sim.load_inductance(results)

print("Inductance matrix (pH):")
print((inductance.inductance_frame() * 1e12).round(3))
print("\nCoupling coefficients:")
print(inductance.coupling_frame().round(4))
print("\nConsistency checks:", inductance.problems() or "ok")

In [ ]:
pairs = inductance.pairs()
pairs["inductance_pH"] = pairs["inductance_H"] * 1e12
pairs[["first", "second", "inductance_pH", "coupling"]].round(4)

For scale, the Neumann formula for the two centrelines gives M ≈ 19.3 pH. The mutual is positive: the coils are wound
the same way, so their fluxes add. Swapping `plus_port` and `minus_port` of one source flips the sign of its row and
column and leaves the self inductances unchanged.

## Validation: two coils side by side

Two TopMetal2 coils at a 39.35 µm pitch (a 10 µm gap) have a small negative mutual inductance: the return flux of
one coil threads the other backwards. Its references:

- **Centreline filaments.** The Neumann formula over the two coil centrelines gives −1.086 pH. Moving both filaments
  to the inner or the outer edges of the 2 µm traces gives −0.705 and −1.606 pH: the result depends strongly on where
  the current flows across the trace.
- **PEC bundle.** A perfect conductor carries its current on the trace surface, crowded towards the inside of the
  loop and towards the other lead. Each coil then encloses about 6 % less area, which lowers |M| by about 11 %.
  Modelling each trace as filaments on its surface, with the current split that a perfect conductor takes (the one
  of least magnetic energy), gives **−0.960 pH** in open space and L = 52.53 pH. This reference was computed offline
  and is quoted here.
- **Outer wall.** The PEC outer wall acts as an image current behind each of the six faces of the domain. One image
  per face, with the planes read from the mesh, gives the factor that makes the reference comparable with the run.

The check passes when Palace is within 3 % of the PEC-bundle reference on both M and L. The mesh is finer here (0.5
µm at the metal) and the solve uses the default Order 2.

In [ ]:
pair = MagnetostaticSim()
pair.set_output_dir("./palace-sim-magnetostatic-pair")
pair.set_geometry(inductor2_pair(pitch=39.35))
pair.set_stack()
pair.set_airbox(margin_x=100, margin_y=100, z_above=150, z_below=150)
pair.add_current_source("A", plus_port="A_P1", minus_port="A_P2")
pair.add_current_source("B", plus_port="B_P1", minus_port="B_P2")
pair.mesh(refined_mesh_size=0.5, max_mesh_size=30.0)

pair_inductance = pair.load_inductance(pair.run(check_cache=True))
print((pair_inductance.inductance_frame() * 1e12).round(4))
print("Consistency checks:", pair_inductance.problems() or "ok")

The filament reference: the Neumann double integral of dl·dl′/r over two closed paths, with Gauss-Legendre points on
pieces of at most 1 µm. `offset` shifts a path towards the loop interior (the inner trace edge is 1 µm in), `mirror`
reflects a path in a wall, and `wall_factor` adds one image per wall (a PEC wall reverses the mirrored current).

In [ ]:
CENTRE_Y, APOTHEM, LEAD_X, Z_MID = 44.675, 13.675, 2.05, 12.53  # um, TopMetal2


def offset(path, u):
    """Shift an open, clockwise polyline by u um towards the loop interior."""
    t = np.diff(path, axis=0)
    t /= np.linalg.norm(t, axis=1)[:, None]
    n = np.c_[t[:, 1], -t[:, 0]]
    out = path.copy()
    out[0] += u * n[0]
    out[-1] += u * n[-1]
    for k in range(1, len(path) - 1):
        out[k] += u * (n[k - 1] + n[k]) / (1 + n[k - 1] @ n[k])
    return out


def coil(x0=0.0, u=0.0):
    """Closed current path of one coil, P1 -> octagon -> P2 -> P1, as 3D points (um)."""
    r = APOTHEM / np.cos(np.pi / 8)
    angles = np.deg2rad(22.5 + 45 * np.array([5, 4, 3, 2, 1, 0, 7, 6]))
    octagon = np.c_[r * np.cos(angles), CENTRE_Y + r * np.sin(angles)]
    y0 = CENTRE_Y - APOTHEM
    leads = np.array([(-LEAD_X, 0), (-LEAD_X, y0), *octagon, (LEAD_X, y0), (LEAD_X, 0)])
    path = offset(leads, u)
    path = np.c_[path[:, 0] + x0, path[:, 1], np.full(len(path), Z_MID)]
    return np.vstack([path, path[:1]])


def quadrature(path, h=1.0):
    """Gauss-Legendre points and weighted tangents, 4 per piece of at most h um."""
    x, w = np.polynomial.legendre.leggauss(4)
    points, tangents = [], []
    for a, b in zip(path[:-1], path[1:], strict=True):
        k = int(np.ceil(np.linalg.norm(b - a) / h))
        t = ((np.arange(k)[:, None] + (x + 1) / 2) / k).ravel()
        points.append(a + t[:, None] * (b - a))
        tangents.append(np.tile(w / (2 * k), k)[:, None] * (b - a))
    return np.vstack(points), np.vstack(tangents)


def neumann(a, b):
    """Mutual inductance of two closed filaments in um, in H."""
    (pa, da), (pb, db) = quadrature(a), quadrature(b)
    r = np.linalg.norm(pa[:, None] - pb[None], axis=-1)
    return 1e-7 * 1e-6 * np.sum((da @ db.T) / r)  # mu0/4pi, um -> m


def mirror(path, axis, plane):
    """Reflect a path in the plane x[axis] = plane."""
    out = path.copy()
    out[:, axis] = 2 * plane - out[:, axis]
    return out


def wall_factor(a, b, bbox, pec=True):
    """Relative change of the mutual from one image of b in each wall of bbox."""
    sign = -1.0 if pec else 1.0
    walls = [
        (axis, bbox[f"{name}{side}"])
        for axis, name in enumerate("xyz")
        for side in ("min", "max")
    ]
    images = sum(neumann(a, mirror(b, axis, plane)) for axis, plane in walls)
    return sign * images / neumann(a, b)

In [ ]:
PITCH = 39.35
M_BUNDLE = -0.960e-12  # H, PEC bundle, open space
L_BUNDLE = 52.51e-12  # H, PEC bundle in this PEC box (open space: 52.53 pH)

a, b = coil(), coil(PITCH)
m_centreline = neumann(a, b)
m_inner = neumann(coil(u=1.0), coil(PITCH, u=1.0))
m_outer = neumann(coil(u=-1.0), coil(PITCH, u=-1.0))

bbox = json.loads(Path(pair.output_dir, "metadata.json").read_text())["mesh"]["bbox"]
delta = wall_factor(a, b, bbox, pec=True)
m_ref = M_BUNDLE * (1 + delta)

m_palace = pair_inductance.mutual_inductance("A", "B")
error_m = m_palace / m_ref - 1
in_bracket = min(m_inner, m_outer) < m_palace < max(m_inner, m_outer)
errors_l = [pair_inductance.self_inductance(name) / L_BUNDLE - 1 for name in "AB"]

pH = 1e12
print(f"Centreline Neumann:         {m_centreline * pH:.4f} pH")
print(f"Edge bracket (inner/outer): {m_inner * pH:.4f} / {m_outer * pH:.4f} pH")
print(f"PEC wall images:            {delta:+.2%}")
print(f"PEC bundle reference:       {m_ref * pH:.4f} pH")
print(f"Palace M_AB:                {m_palace * pH:.4f} pH")
print(f"  vs PEC bundle:            {error_m:+.2%}")
print(f"  inside the edge bracket:  {in_bracket}")
print(f"  vs centreline (+ images): {m_palace / (m_centreline * (1 + delta)) - 1:+.2%}")
print(f"Palace L_A, L_B vs bundle:  {errors_l[0]:+.2%}, {errors_l[1]:+.2%}")
print(f"Coupling k_AB:              {pair_inductance.coupling('A', 'B'):.4f}")

assert abs(error_m) <= 0.03
assert all(abs(error) <= 0.03 for error in errors_l)

Palace's self inductances lie about 2 % below the bundle, as they must: a conforming finite-element solution
underestimates the stored energy, and refining the mesh raises them. The centreline formula, by contrast,
overestimates |M| by about 10 %: it puts the current on the trace centre rather than on the crowded surface.

## Notes

- **Outer wall bracket.** `MagnetostaticSim(metallic_boundaries=False)` makes the outer wall a perfect magnetic
  conductor. A PEC and a PMC run bracket the open-space value: an energy bound for the self inductances, a
  first-order bracket for the mutuals. Their mean cancels the first-order wall error.
- **Mesh convergence.** With Palace v0.18.1 at Order 2, refining the conductor mesh of the side-by-side pair from
  0.7 to 0.5 µm changed M_AB by 0.18 % and the self inductances by 0.4 %.